In [1]:
# Install needed libraries
!pip install transformers datasets sentencepiece nltk scikit-learn --quiet


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.4/491.4 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 19.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2025.3.0 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cu

In [2]:
!pip install sentence-transformers language-tool-python evaluate bert_score --quiet



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.7/54.7 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.1/55.1 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 57.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 51.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 44.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.

In [3]:
import pandas as pd
from tqdm import tqdm
from sentence_transformers import SentenceTransformer, util
import torch
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
import evaluate
from bert_score import score as bert_score



In [4]:
# import pandas as pd
# import nltk
# from sklearn.model_selection import train_test_split
# from transformers import pipeline

# nltk.download('punkt')




In [5]:
# Load your CSV (adjust path if needed)
df = pd.read_csv('/content/ASAP2_train_sourcetexts.csv', on_bad_lines='skip', engine='python')
df = df.reset_index(drop=True)

final_df = df.sample(frac=0.06, random_state=42).reset_index(drop=True)

final_df = final_df[['essay_id', 'full_text', 'score']]
# Display first few rows
final_df = final_df.reset_index(drop=True)
print(f"Total records: {len(final_df)}")
final_df.head()

Total records: 1484


,essay_id,full_text,score
0,AAAOPP13416000044958,Driverless cars are a good inovention for the ...,3
1,AAAOPP13416000082685,I think what they are trying to say is that be...,2
2,AAAOPP13416000071582,Have you ever wanted to go around the world he...,2
3,5027982,The advantages of limitng car use can be a ben...,3
4,AAAOPP13416000043977,"In 1976 , NASA took a picture of the Cydonia a...",2


In [6]:
df.loc[0]

,0
essay_id,AAAVUP14319000159574
score,4
full_text,The author suggests that studying Venus is wor...
assignment,"In ""The Challenge of Exploring Venus,"" the aut..."
prompt_name,Exploring Venus
economically_disadvantaged,Economically disadvantaged
student_disability_status,Identified as having disability
ell_status,No
race_ethnicity,Black/African American
gender,F


In [7]:
# Map numeric score to low/medium/high quality
def score_to_quality(score):
    if score <= 1:
        return 'low'
    elif score <= 3:
        return 'medium'
    else:
        return 'high'

final_df['quality'] = final_df['score'].apply(score_to_quality)


In [8]:
# Feature extractor: number of words and number of sentences
def extract_features(text):
    num_words = len(text.split())
    num_sentences = text.count('.') + text.count('!') + text.count('?')
    return [num_words, num_sentences]

# Apply feature extraction
features = []
for essay in tqdm(final_df['full_text'], desc="Extracting features"):
    features.append(extract_features(essay))

features_df = pd.DataFrame(features, columns=['num_words', 'num_sentences'])
final_df = pd.concat([final_df, features_df], axis=1)


Extracting features: 100%|██████████| 1484/1484 [00:00<00:00, 9916.70it/s]


In [9]:
# 80% Train, 20% Validation
train_df, val_df = train_test_split(final_df, test_size=0.2, random_state=42)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)


print(f"Train Essays: {len(train_df)}, Validation Essays: {len(val_df)}")


Train Essays: 1187, Validation Essays: 297


In [10]:
X_train = train_df[['num_words', 'num_sentences']]
y_train = train_df['quality']

X_val = val_df[['num_words', 'num_sentences']]
y_val = val_df['quality']

# Train classifier
clf = LogisticRegression(max_iter=500)
clf.fit(X_train, y_train)

# Check accuracy
print(f"Training Accuracy: {clf.score(X_train, y_train):.2f}")
print(f"Validation Accuracy: {clf.score(X_val, y_val):.2f}")


Training Accuracy: 0.77
Validation Accuracy: 0.80


In [11]:
# Feedback templates
feedback_low = [
    "Focus on organizing your ideas more clearly.",
    "Work on expanding your main points with supporting details.",
    "Improve grammar and sentence structure for better clarity.",
    "Make sure your essay addresses all parts of the prompt."
]

feedback_medium = [
    "Your ideas are clear; now work on connecting them more smoothly.",
    "Add more examples to support your arguments.",
    "Improve transitions between paragraphs to strengthen coherence.",
    "Work on polishing grammar and varying sentence structure."
]

feedback_high = [
    "Excellent work! Consider enhancing your arguments with more advanced examples.",
    "Great essay! You could further strengthen it by refining transitions.",
    "Very good structure and grammar! Pay attention to minor details for even greater clarity.",
    "Strong essay! Push for deeper analysis in your next draft."
]


In [12]:
# Load models
model = SentenceTransformer('all-MiniLM-L6-v2')



/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [13]:
def generate_feedback_model(essay_text):
    # Extract features
    features = extract_features(essay_text)

    # Predict quality
    predicted_quality = clf.predict([features])[0]

    # Select appropriate feedback set
    if predicted_quality == 'low':
        feedback_list = feedback_low
    elif predicted_quality == 'medium':
        feedback_list = feedback_medium
    else:
        feedback_list = feedback_high

    # Embedding matching
    feedback_embeddings = model.encode(feedback_list, convert_to_tensor=True)
    essay_embedding = model.encode(essay_text, convert_to_tensor=True)

    cosine_scores = util.cos_sim(essay_embedding, feedback_embeddings)
    best_idx = torch.argmax(cosine_scores)

    return feedback_list[best_idx], predicted_quality


In [14]:
# Generate feedback for validation essays
generated_feedbacks = []
predicted_qualities = []

for idx, row in tqdm(val_df.iterrows(), total=len(val_df), desc="Generating validation feedbacks"):
    essay = row['full_text']
    feedback, quality = generate_feedback_model(essay)
    generated_feedbacks.append(feedback)
    predicted_qualities.append(quality)

# Save to validation dataframe
val_df['predicted_quality'] = predicted_qualities
val_df['generated_feedback'] = generated_feedbacks


Generating validation feedbacks:   0%|          | 0/297 [00:00<?, ?it/s]/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
Generating validation feedbacks:   0%|          | 1/297 [00:00<03:44,  1.32it/s]/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739

In [15]:
# Test with a new essay
input_essay = """
Space exploration is vital for understanding our universe. Discoveries like Mars rovers have helped humanity learn about planetary atmospheres and geology. New missions can teach us even more.
"""

feedback, quality = generate_feedback_model(input_essay)

print("\n--- Essay Feedback ---")
print(f"Predicted Quality: {quality}")
print("\nGenerated Feedback:")
print(feedback)


/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(



--- Essay Feedback ---
Predicted Quality: medium

Generated Feedback:
Your ideas are clear; now work on connecting them more smoothly.


In [16]:
!pip install gradio --quiet


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.1/54.1 MB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.9/322.9 kB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 130.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 kB 6.2 MB/s eta 0:00:00


In [17]:
import gradio as gr

def gradio_interface(essay_text):
    feedback, quality = generate_feedback_model(essay_text)
    return f"Predicted Quality: {quality}\n\nFeedback: {feedback}"

iface = gr.Interface(
    fn=gradio_interface,
    inputs=gr.Textbox(lines=6, placeholder="Paste your essay here...", label="Paste Your Essay Here"),
    outputs=gr.Textbox(label="Generated Feedback"),
    title="📝 Automated Essay Feedback Generator",
    description="This app gives AI-generated feedback on your essay along with a predicted quality level.",
    theme="default"
)

iface.launch()


It looks like you are running Gradio on a hosted a Jupyter notebook. For the Gradio app to work, sharing must be enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c4b161c906316b83e8.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [18]:
val_df.head()

,essay_id,full_text,score,quality,num_words,num_sentences,predicted_quality,generated_feedback
0,AAAVUP14319000052104,Sometimes there are authors that don't ever su...,2,medium,176,14,medium,Improve transitions between paragraphs to stre...
1,5184441,"Dear Mr./Mrs. Senator,\n\nIn light of previous...",4,high,485,20,medium,Add more examples to support your arguments.
2,AAAOPP13416000001493,I believe that driverless cars should become a...,4,high,548,19,high,Excellent work! Consider enhancing your argume...
3,AAAOPP13416000000058,This formation is not evidence of alians. We h...,2,medium,174,9,medium,Add more examples to support your arguments.
4,AAATRP14318000691471,"In the article ""Making Mona Lisa Smiles,"" the ...",5,high,568,36,high,Great essay! You could further strengthen it b...


In [19]:
from transformers import AutoTokenizer
from bert_score import score as bert_score

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained('roberta-large')

# === Cleaning function ===
def basic_clean(text):
    """Remove line breaks and excess spacing."""
    text = text.replace('\n', ' ')
    return ' '.join(text.split())

# === Token length check ===
def is_within_token_limit(text, max_tokens=512):
    tokens = tokenizer.tokenize(text)
    return len(tokens) <= max_tokens

# === Optional truncation if you want to keep longer texts ===
def truncate_to_max_tokens(text, max_tokens=512):
    tokens = tokenizer.tokenize(text)
    if len(tokens) > max_tokens:
        tokens = tokens[:max_tokens]
    return tokenizer.convert_tokens_to_string(tokens)

# === Step 1: Clean input ===
val_df['full_text_clean'] = val_df['full_text'].astype(str).apply(basic_clean)
val_df['generated_feedback_clean'] = val_df['generated_feedback'].astype(str).apply(basic_clean)

# === Step 2: Filter valid rows ===
val_df['valid_for_bert'] = val_df.apply(
    lambda row: is_within_token_limit(row['full_text_clean']) and
                is_within_token_limit(row['generated_feedback_clean']), axis=1
)

# === Step 3: Keep only valid samples ===
val_df_safe = val_df[val_df['valid_for_bert']].reset_index(drop=True)

print(f"✅ Original essays: {len(val_df)}")
print(f"✅ Essays safe for BERTScore evaluation: {len(val_df_safe)}")

# === Optional: truncate long examples instead of dropping ===
# val_df_safe['full_text_clean'] = val_df_safe['full_text_clean'].apply(lambda x: truncate_to_max_tokens(x, 512))
# val_df_safe['generated_feedback_clean'] = val_df_safe['generated_feedback_clean'].apply(lambda x: truncate_to_max_tokens(x, 512))

# === Step 4: Run BERTScore ===
P, R, F1 = bert_score(
    cands=val_df_safe['generated_feedback_clean'].tolist(),
    refs=val_df_safe['full_text_clean'].tolist(),
    lang="en",
    verbose=True
)

# === Step 5: Print average results ===
print("\n--- 🔍 BERTScore Evaluation ---")
print(f"Precision: {P.mean().item():.4f}")
print(f"Recall:    {R.mean().item():.4f}")
print(f"F1 Score:  {F1.mean().item():.4f}")



tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

...

calculating scores...
computing bert embedding.


  0%|          | 0/4 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/4 [00:00<?, ?it/s]

done in 10.58 seconds, 19.66 sentences/sec

--- 🔍 BERTScore Evaluation ---
Precision: 0.8410
Recall:    0.7728
F1 Score:  0.8054
